# Week 10 - RAG (Retrieval Augmented Generation)
**Using the Gemini API instead of OpenAI**

This notebook follows the Lab 10 instructions, with `ChatOpenAI` replaced by `ChatGoogleGenerativeAI` (Gemini) via LangChain's Google GenAI integration.

Make sure the `company_docs/` folder (with `hr_policy.txt`, `benefits.txt`, `it_policy.txt`) sits in the same folder as this notebook before running.


## Part 1: Document Loading
### Task 1.1: Load Documents

In [ ]:
import os
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

# Load environment variables (GEMINI_API_KEY)
load_dotenv()

# Load all .txt files from company_docs/ directly with plain Python
# (avoids depending on the deprecated langchain-community DirectoryLoader/TextLoader,
# which are just simple filesystem readers with no external provider anyway)
docs_folder = 'company_docs/'
documents = []

for filename in sorted(os.listdir(docs_folder)):
    if filename.endswith('.txt'):
        filepath = os.path.join(docs_folder, filename)
        with open(filepath, 'r', encoding='utf-8') as f:
            text = f.read()
        documents.append(Document(page_content=text, metadata={'source': filepath}))

print(f'Loaded {len(documents)} documents')
print(f'First doc preview: {documents[0].page_content[:200]}...')


### Task 1.2: Split into Chunks

In [ ]:
# Create text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,       # Characters per chunk
    chunk_overlap=50,     # Overlap between chunks
    length_function=len,
    separators=['\n\n', '\n', '. ', ' ', '']
)

# Split documents
chunks = text_splitter.split_documents(documents)
print(f'Split into {len(chunks)} chunks')

print('\nSample chunks:')
for i, chunk in enumerate(chunks[:3]):
    print(f'\nChunk {i+1}:')
    print(chunk.page_content)
    print(f'Length: {len(chunk.page_content)} chars')


**Understanding Parameters:**
- `chunk_size`: Target size for each chunk
- `chunk_overlap`: Characters shared between chunks (preserves context across a split)
- `separators`: Tries to split at natural boundaries (paragraphs, then sentences, then words)


## Part 2: Simple Retrieval
### Task 2.1: Build Keyword Search

In [ ]:
def simple_search(query, chunks, top_k=3):
    """
    Simple keyword-based search.
    Returns top_k most relevant chunks.
    """
    query_lower = query.lower()

    # Score each chunk
    scored_chunks = []
    for chunk in chunks:
        content_lower = chunk.page_content.lower()
        # Count keyword matches
        score = 0
        for word in query_lower.split():
            score += content_lower.count(word)
        if score > 0:
            scored_chunks.append((score, chunk))

    # Sort by score and return top k
    scored_chunks.sort(reverse=True, key=lambda x: x[0])
    return [chunk for score, chunk in scored_chunks[:top_k]]


# Test it
query = 'What is the vacation policy?'
relevant = simple_search(query, chunks)
print(f'Found {len(relevant)} relevant chunks:')
for i, chunk in enumerate(relevant):
    print(f'\n--- Chunk {i+1} ---')
    print(chunk.page_content)


### Task 2.2: Test Different Queries

In [ ]:
# Test multiple queries
test_queries = [
    'How many vacation days do employees get?',
    'What is the remote work policy?',
    'Tell me about parental leave',
]

for query in test_queries:
    print(f'\nQuery: {query}')
    results = simple_search(query, chunks, top_k=2)
    print(f'Found {len(results)} relevant chunks')
    if results:
        print(f'Top result: {results[0].page_content[:100]}...')


## Part 3: RAG Pipeline
### Task 3.1: Build RAG Function

In [ ]:
# Initialize Gemini via LangChain
llm = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash',
    google_api_key=os.getenv('GEMINI_API_KEY'),
    temperature=0   # Deterministic for factual answers
)


def rag_query(query, chunks, top_k=3):
    """
    RAG pipeline: Retrieve -> Generate
    """
    # Step 1: Retrieve relevant chunks
    relevant_chunks = simple_search(query, chunks, top_k)

    if not relevant_chunks:
        return 'No relevant information found in documents.'

    # Step 2: Build context
    context = '\n\n---\n\n'.join([
        chunk.page_content for chunk in relevant_chunks
    ])

    # Step 3: Create prompt
    prompt = f'''You are a helpful assistant. Answer the question using ONLY the context provided below.
If the answer is not in the context, say so.

Context:
{context}

Question: {query}

Answer:'''

    # Step 4: Generate answer
    response = llm.invoke(prompt)
    return response.content


### Task 3.2: Test RAG System

In [ ]:
# Test questions
questions = [
    'How many vacation days do full-time employees get?',
    'Can employees work from home?',
    'What is the parental leave policy?',
    'What is the dress code?'  # Not in docs
]

for question in questions:
    print(f'\n{"="*60}')
    print(f'Q: {question}')
    print(f'{"="*60}')
    answer = rag_query(question, chunks)
    print(f'A: {answer}')


**Expected Behavior:** Questions 1-3 should get accurate answers pulled from the documents. Question 4 (dress code) isn't covered in any of the sample docs, so the model should say the answer isn't in the context.


## Bonus: Compare With vs Without RAG

In [ ]:
def ask_without_rag(question):
    """
    Ask Gemini directly (no retrieved context)
    """
    messages = [
        {'role': 'system', 'content': 'You are a helpful HR assistant.'},
        {'role': 'user', 'content': question}
    ]
    response = llm.invoke(messages)
    return response.content


# Compare
question = 'How many vacation days do employees get?'

print('WITHOUT RAG:')
print(ask_without_rag(question))

print('\nWITH RAG:')
print(rag_query(question, chunks))


**Notice:** Without RAG, Gemini gives a generic, made-up-sounding answer since it has no knowledge of your company's actual policy. With RAG, the answer is specific to YOUR documents (15 days) — this is the entire point of Retrieval Augmented Generation.

---
### Next steps
Coming up: replacing this simple keyword search with real **vector embeddings and semantic search**, so the retrieval step understands meaning, not just literal word overlap.